# 📓 Day 3: Advanced Python (OOP, Exception Handling & File I/O)

Welcome to the Day 3 Coding Notebook! Today, we transition from writing functional scripts to building robust, object-oriented, persistent, and crash-proof Python systems. These core skills form the basis for state tracking in AI agents, fallback mechanisms in LLM pipelines, and system tool integration.

## 🎯 Learning Objectives
1. Define **Classes** and instantiate **Objects** with custom attributes and methods.
2. Implement the **Four Pillars of OOP** (Encapsulation, Inheritance, Polymorphism, Abstraction).
3. Intercept runtime errors gracefully using **Exception Handling** (`try-except-else-finally`).
4. Design and raise **Custom Exception classes** for business logic validation.
5. Read, write, and append local files using **Context Managers** (`with` blocks) for text, JSON, and CSV data.
6. Build a complete **E-commerce Inventory Manager** and a production-grade **Transaction Ledger System**.

---
## 1. Object-Oriented Programming (OOP) Basics
Let's write a simple class to represent a `Saree` inventory item. We will explore initializers, instance variables, class variables, and magic (dunder) methods.

In [ ]:
class Saree:
    # Class variable: Shared across ALL instances
    gst_rate = 0.05

    def __init__(self, design_code: str, fabric: str, price: float):
        # Instance variables: Unique to each saree object
        self.design_code = design_code
        self.fabric = fabric
        self.price = price
        self.stock = 0

    # Instance method
    def update_stock(self, quantity: int):
        self.stock += quantity
        print(f"Stock updated for {self.design_code}: {self.stock} items available.")

    # Instance method incorporating class variable
    def calculate_taxed_price(self) -> float:
        return self.price + (self.price * Saree.gst_rate)

    # Dunder method: User-friendly string representation
    def __str__(self) -> str:
        return f"{self.fabric.title()} Saree (Code: {self.design_code}) - Price: ₹{self.price:.2f} | Stock: {self.stock}"

    # Dunder method: Developer debugging representation
    def __repr__(self) -> str:
        return f"Saree(design_code='{self.design_code}', fabric='{self.fabric}', price={self.price})"


# 1. Create instances (Objects)
saree1 = Saree(design_code="KANCHI-001", fabric="silk", price=3500.0)
saree2 = Saree(design_code="COT-052", fabric="cotton", price=1200.0)

# 2. Call methods and view printouts
saree1.update_stock(15)
print(saree1)
print(f"Tax-inclusive price: ₹{saree1.calculate_taxed_price():.2f}")

print(saree2)
print(repr(saree2))

---
## 2. The Four Pillars of OOP
Let's see how Python implements **Encapsulation**, **Inheritance**, **Polymorphism**, and **Abstraction**.

In [ ]:
from abc import ABC, abstractmethod

# --- 1. ABSTRACTION ---
# Define an Abstract Base Class. It cannot be instantiated directly.
class User(ABC):
    def __init__(self, name: str, user_id: str):
        self.name = name
        self.user_id = user_id

    @abstractmethod
    def get_role_details(self) -> str:
        """Abstract method: Child classes MUST implement this."""
        pass

# --- 2. INHERITANCE ---
class Customer(User):
    def __init__(self, name: str, user_id: str, credit_limit: float):
        super().__init__(name, user_id) # Trigger parent constructor
        # --- 3. ENCAPSULATION ---
        # Private balance variable (name-mangled)
        self.__balance = 0.0
        self.credit_limit = credit_limit

    # Getter for balance
    def get_balance(self) -> float:
        return self.__balance

    # Setter with business validation rules
    def update_balance(self, amount: float):
        if self.__balance + amount < -self.credit_limit:
            print(f"❌ Transaction Rejected: Exceeds credit limit of ₹{self.credit_limit}")
        else:
            self.__balance += amount
            print(f"✅ Balance updated for {self.name}. Current: ₹{self.__balance:.2f}")

    # Implement Abstract Method (Polymorphism part A)
    def get_role_details(self) -> str:
        return f"Customer Account: {self.name} (ID: {self.user_id}) | Credit Limit: ₹{self.credit_limit}"


class Clerk(User):
    def __init__(self, name: str, user_id: str, counter_number: int):
        super().__init__(name, user_id)
        self.counter_number = counter_number

    # Implement Abstract Method differently (Polymorphism part B)
    def get_role_details(self) -> str:
        return f"Clerk Staff: {self.name} (ID: {self.user_id}) | Assigned Counter: {self.counter_number}"


# 1. Instantiation checks
try:
    # This will fail because User is abstract and cannot be directly built
    guest = User("Anonymous", "U000")
except TypeError as e:
    print(f"❌ Cannot instantiate abstract class User directly: {e}\n")

cust1 = Customer("Ramesh Kumar", "C101", credit_limit=5000.0)
staff1 = Clerk("Vijay Prasad", "S205", counter_number=3)

# 2. Polymorphism demonstration
users_list = [cust1, staff1]
for u in users_list:
    print(u.get_role_details())

print("")
# 3. Encapsulation testing
cust1.update_balance(-4000.0) # Withdraw money
cust1.update_balance(-2000.0) # Overdraft check - should fail

try:
    # Attempting to modify raw balance directly raises error because it's private
    cust1.__balance = 9999.0
    print(f"Direct Balance modification attempt resulted in: {cust1.get_balance()} (Check didn't write to private variable)")
except AttributeError as ae:
    print(f"❌ Blocked direct access: {ae}")

---
## 3. Exception Handling
Let's see how python handles division by zero, invalid input formatting, and how we define custom exceptions to handle complex domain checks.

In [ ]:
# Define custom validation exceptions
class InvalidBarcodeError(Exception):
    """Raised when saree barcode format is incorrect."""
    pass

class SareeOutOfStockError(Exception):
    """Raised when trying to sell an item with 0 stock."""
    pass

def process_saree_sale(barcode: str, stock_level: int) -> float:
    # Validate input parameters
    if not barcode.startswith("SR-") or len(barcode) != 8:
        raise InvalidBarcodeError(f"Barcode '{barcode}' is invalid! Must match 'SR-XXXXX' format.")
        
    if stock_level <= 0:
        raise SareeOutOfStockError("This design is completely out of stock!")
        
    sale_price = 1500.0
    return sale_price

# Test Try-Except Flow
barcodes_to_test = ["SR-12345", "INVALID_CODE", "SR-99999"]
stocks_to_test = [5, 10, 0]

for b, s in zip(barcodes_to_test, stocks_to_test):
    print(f"\nAttempting sale: Barcode={b}, Stock={s}")
    try:
        price = process_saree_sale(b, s)
        print(f"🎉 Sale Success! Item price: ₹{price:.2f}")
    except InvalidBarcodeError as ibe:
        print(f"❌ Scanning Error: {ibe}")
    except SareeOutOfStockError as sose:
        print(f"❌ Inventory Error: {sose}")
    else:
        print("✅ No errors encountered during transaction calculations.")
    finally:
        print("🔄 Scanner sensor state reset.")

---
## 4. File Handling (Text, JSON, and CSV)
Let's see how we read and write configuration files and record list logs permanently.

In [ ]:
import json
import csv
import os

# 1. TEXT FILES
text_filepath = "session_notes.txt"
print("--- 1. Writing Text File ---")
with open(text_filepath, "w") as f:
    f.write("Advanced Python Course Notes\n")
    f.write("Topic: File input/output systems\n")

with open(text_filepath, "r") as f:
    content = f.read()
    print(f"File Contents:\n{content}")

# 2. JSON FILES (For complex structured dictionary database maps)
json_filepath = "database.json"
db_data = {
    "version": "1.0.0",
    "restaurants": [
        {"name": "Biryani House", "location": "Nagpur", "rating": 4.5},
        {"name": "Ganga Cafe", "location": "Coimbatore", "rating": 4.2}
    ]
}

print("\n--- 2. Writing JSON File ---")
with open(json_filepath, "w") as f:
    json.dump(db_data, f, indent=4) # Serializes dict to file
    print("Saved dict successfully to JSON.")

with open(json_filepath, "r") as f:
    loaded_db = json.load(f) # Deserializes file to dict
    print(f"Loaded Cafe Name: {loaded_db['restaurants'][1]['name']}")

# 3. CSV FILES (For spreadsheet-like tabular data export)
csv_filepath = "reports.csv"
csv_headers = ["Date", "Item", "Amount"]
csv_rows = [
    ["2026-06-06", "Saree Type A", 1500.0],
    ["2026-06-06", "Saree Type B", 2400.0],
    ["2026-06-07", "Silk Thread", 350.0]
]

print("\n--- 3. Writing CSV File ---")
with open(csv_filepath, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(csv_headers) # Header
    writer.writerows(csv_rows)   # Data rows
    print("CSV Report compiled.")

with open(csv_filepath, "r") as f:
    reader = csv.reader(f)
    print("Reading CSV Data:")
    for row in reader:
        print(row)

# Cleanup generated temporary files
for path in [text_filepath, json_filepath, csv_filepath]:
    if os.path.exists(path):
        os.remove(path)

---
## 🛠️ Mini Project: E-commerce Inventory System

### Scenario
We need to build a modular console-based inventory system. The system uses a class `InventoryItem` to store SKU metadata and a class `Warehouse` to manage stock collections. It must persist catalog logs to a local JSON file (`inventory_db.json`) and gracefully catch inventory anomalies.

In [ ]:
import json
import os

# Custom Exceptions
class StockShortageError(Exception):
    """Raised when request exceeds inventory quantity."""
    pass

class ItemNotFoundError(Exception):
    """Raised when looking up SKU not present in catalog."""
    pass

class InventoryItem:
    def __init__(self, sku: str, name: str, price: float, quantity: int = 0):
        self.sku = sku
        self.name = name
        self.price = price
        self.quantity = quantity

    def add_stock(self, count: int):
        self.quantity += count

    def deduct_stock(self, count: int):
        if count > self.quantity:
            raise StockShortageError(f"Shortage! SKU {self.sku} has {self.quantity} items; requested {count}.")
        self.quantity -= count

    def to_dict(self) -> dict:
        return {
            "sku": self.sku,
            "name": self.name,
            "price": self.price,
            "quantity": self.quantity
        }

    def __str__(self) -> str:
        return f"{self.name} (SKU: {self.sku}) | Price: ₹{self.price:.2f} | Stock: {self.quantity}"


class Warehouse:
    def __init__(self, db_file: str = "inventory_db.json"):
        self.db_file = db_file
        self.items = {}  # SKU string -> InventoryItem object
        self.load_from_disk()

    def load_from_disk(self):
        if not os.path.exists(self.db_file):
            return
        try:
            with open(self.db_file, "r") as f:
                raw_data = json.load(f)
                for sku, info in raw_data.items():
                    self.items[sku] = InventoryItem(
                        sku=info["sku"],
                        name=info["name"],
                        price=info["price"],
                        quantity=info["quantity"]
                    )
        except (json.JSONDecodeError, IOError) as e:
            print(f"⚠️ Disk Load Error: {e}. Resetting database state.")

    def save_to_disk(self):
        try:
            serializable_dict = {sku: item.to_dict() for sku, item in self.items.items()}
            with open(self.db_file, "w") as f:
                json.dump(serializable_dict, f, indent=4)
        except IOError as e:
            print(f"❌ Critical: Failed to save changes to hard drive: {e}")

    def add_item_to_catalog(self, sku: str, name: str, price: float):
        if sku in self.items:
            print(f"Item SKU {sku} already exists. Incrementing catalog metadata.")
        else:
            self.items[sku] = InventoryItem(sku, name, price)
            self.save_to_disk()
            print(f"✅ Successfully registered: {name}")

    def receive_shipment(self, sku: str, count: int):
        if sku not in self.items:
            raise ItemNotFoundError(f"Cannot add stock! SKU '{sku}' not registered in system.")
        self.items[sku].add_stock(count)
        self.save_to_disk()

    def fulfill_order(self, sku: str, count: int):
        if sku not in self.items:
            raise ItemNotFoundError(f"Cannot fulfill order! SKU '{sku}' not registered in system.")
        self.items[sku].deduct_stock(count)
        self.save_to_disk()

    def print_report(self):
        print("\n====== WAREHOUSE STOCK STATUS ======")
        for item in self.items.values():
            print(item)
        print("=====================================")


# --- TEST SYSTEM RUN ---
wh = Warehouse("temp_inventory.json")

# 1. Add catalog items
wh.add_item_to_catalog("SKU-SHIRT-M", "Premium Cotton Shirt (M)", 1500.0)
wh.add_item_to_catalog("SKU-JEANS-32", "Denim Slim Fit Jeans (32)", 2200.0)

# 2. Receive stock shipments
try:
    wh.receive_shipment("SKU-SHIRT-M", 50)
    wh.receive_shipment("SKU-JEANS-32", 20)
    # Try adding stock to unregistered item
    wh.receive_shipment("SKU-SHOES-9", 10)
except ItemNotFoundError as infe:
    print(f"❌ Error Blocked: {infe}")

# 3. Process orders with shortages
try:
    wh.fulfill_order("SKU-SHIRT-M", 5)   # Should succeed
    wh.fulfill_order("SKU-JEANS-32", 25)  # Should raise StockShortageError
except (ItemNotFoundError, StockShortageError) as error:
    print(f"❌ Order Fulfillment Failed: {error}")

# 4. View Report
wh.print_report()

# Cleanup DB file
if os.path.exists("temp_inventory.json"):
    os.remove("temp_inventory.json")

---
## 🚀 Industry Project: Transaction Ledger Engine

### Scenario
A fintech startup in Coimbatore needs a robust Transaction Ledger Engine. The system must process transactions (debits and credits), enforce accounts balances rules, handle concurrency and corruption errors via exceptions, load starting state from a JSON file, and compile a CSV audit report for bank regulators.

In [ ]:
import json
import csv
import os
from datetime import datetime

# Domain Custom Errors
class InvalidTransactionError(Exception):
    """Base exception for invalid transactions."""
    pass

class OverdraftLimitExceededError(InvalidTransactionError):
    """Raised when account exceeds permitted overdraft limit."""
    pass

class AccountLockedError(InvalidTransactionError):
    """Raised when transaction attempted on a locked security account."""
    pass


class LedgerAccount:
    def __init__(self, account_id: str, holder: str, balance: float = 0.0, max_overdraft: float = 1000.0):
        self.account_id = account_id
        self.holder = holder
        self.__balance = balance # Encapsulated balance
        self.max_overdraft = max_overdraft
        self.is_locked = False

    def get_balance(self) -> float:
        return self.__balance

    def post_transaction(self, tx_type: str, amount: float):
        if self.is_locked:
            raise AccountLockedError(f"Transaction blocked! Account {self.account_id} is locked for audit.")
        
        if amount <= 0:
            raise InvalidTransactionError("Transaction amount must be greater than zero.")

        if tx_type.lower() == "credit":
            self.__balance += amount
        elif tx_type.lower() == "debit":
            # Overdraft validation check
            if self.__balance - amount < -self.max_overdraft:
                raise OverdraftLimitExceededError(
                    f"Failed: Debit amount of ₹{amount} exceeds overdraft limit of ₹{self.max_overdraft}. "
                    f"Current Balance: ₹{self.__balance}"
                )
            self.__balance -= amount
        else:
            raise InvalidTransactionError(f"Unsupported transaction type: {tx_type}")

    def to_dict(self) -> dict:
        return {
            "account_id": self.account_id,
            "holder": self.holder,
            "balance": self.__balance,
            "max_overdraft": self.max_overdraft,
            "is_locked": self.is_locked
        }


class LedgerEngine:
    def __init__(self, db_path: str = "ledger_db.json"):
        self.db_path = db_path
        self.accounts = {}  # account_id -> LedgerAccount
        self.transaction_history = []  # List of raw dict records
        self.load_ledger_data()

    def load_ledger_data(self):
        if not os.path.exists(self.db_path):
            return
        try:
            with open(self.db_path, "r") as f:
                raw = json.load(f)
                for acc_id, meta in raw.get("accounts", {}).items():
                    acc = LedgerAccount(
                        account_id=meta["account_id"],
                        holder=meta["holder"],
                        balance=meta["balance"],
                        max_overdraft=meta["max_overdraft"]
                    )
                    acc.is_locked = meta["is_locked"]
                    self.accounts[acc_id] = acc
                self.transaction_history = raw.get("transactions", [])
        except (json.JSONDecodeError, IOError) as e:
            print(f"⚠️ Ledger DB read error: {e}. Starting with clean engine state.")

    def save_ledger_data(self):
        try:
            out = {
                "accounts": {acc_id: acc.to_dict() for acc_id, acc in self.accounts.items()},
                "transactions": self.transaction_history
            }
            with open(self.db_path, "w") as f:
                json.dump(out, f, indent=4)
        except IOError as e:
            print(f"❌ Persistent Save Error: {e}")

    def register_account(self, account_id: str, holder: str, balance: float, max_overdraft: float):
        if account_id in self.accounts:
            print(f"Account ID {account_id} already registered.")
            return
        self.accounts[account_id] = LedgerAccount(account_id, holder, balance, max_overdraft)
        self.save_ledger_data()

    def execute_transaction(self, account_id: str, tx_type: str, amount: float):
        if account_id not in self.accounts:
            raise InvalidTransactionError(f"Account {account_id} not found in system.")
        
        account = self.accounts[account_id]
        
        # Execute
        account.post_transaction(tx_type, amount)
        
        # Log to ledger transaction log list
        self.transaction_history.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "account_id": account_id,
            "type": tx_type,
            "amount": amount,
            "resulting_balance": account.get_balance()
        })
        self.save_ledger_data()

    def export_csv_audit_report(self, filepath: str = "audit_report.csv"):
        try:
            with open(filepath, "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=["timestamp", "account_id", "type", "amount", "resulting_balance"])
                writer.writeheader()
                writer.writerows(self.transaction_history)
                print(f"📊 Audit CSV exported successfully to: {filepath}")
        except IOError as e:
            print(f"❌ Failed to compile CSV Audit report: {e}")


# --- TEST SYSTEM RUN ---
engine = LedgerEngine("temp_ledger.json")

# 1. Setup Accounts
engine.register_account("ACC-1001", "Amit Sharma", balance=500.0, max_overdraft=200.0)
engine.register_account("ACC-1002", "Priya Nair", balance=100.0, max_overdraft=0.0)

# 2. Post Transactions (Exceptions Handled)
try:
    engine.execute_transaction("ACC-1001", "credit", 1500.0) # Ok
    engine.execute_transaction("ACC-1001", "debit", 2100.0)  # Exceeds current + overdraft (Limit: -200, proposed: -100) -> Should Fail
except InvalidTransactionError as e:
    print(f"❌ Transaction Rejected: {e}")

try:
    engine.execute_transaction("ACC-1002", "debit", 150.0)   # Overdraft is 0 -> Should Fail
except InvalidTransactionError as e:
    print(f"❌ Transaction Rejected: {e}")

# Lock account 1002 for audit, attempt transaction
engine.accounts["ACC-1002"].is_locked = True
try:
    engine.execute_transaction("ACC-1002", "credit", 200.0)  # Locked -> Should Fail
except InvalidTransactionError as e:
    print(f"❌ Transaction Rejected: {e}")

# 3. Export audit ledger logs to CSV
engine.export_csv_audit_report("temp_audit.csv")

# Cleanup temporary database files
for path in ["temp_ledger.json", "temp_audit.csv"]:
    if os.path.exists(path):
        os.remove(path)